In [3]:
import pandas as pd
from sqlalchemy import create_engine, text

db_path = 'C:/Users/许佳佳/Desktop/UK_Smart_Retail_Data_Ops/database/'
engine = create_engine('sqlite:///' + db_path + 'olist_dw.db')

# **overview**


In [4]:
import pandas as pd
from sqlalchemy import create_engine, text

db_path = 'C:/Users/许佳佳/Desktop/UK_Smart_Retail_Data_Ops/database/'
engine = create_engine('sqlite:///' + db_path + 'olist_dw.db')

# ── 分析1：订单整体概览 ──
overview = """
SELECT
    COUNT(DISTINCT o.order_id)     AS total_orders,
    COUNT(DISTINCT o.customer_id)  AS total_customers,
    COUNT(DISTINCT oi.seller_id)   AS total_sellers,
    ROUND(SUM(oi.price), 2)        AS total_revenue,
    ROUND(AVG(oi.price), 2)        AS avg_item_price,
    MIN(DATE(o.order_purchase_timestamp))  AS earliest_order,
    MAX(DATE(o.order_purchase_timestamp))  AS latest_order
FROM dim_orders o
JOIN fact_order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
"""

df_overview = pd.read_sql(text(overview), engine)
print("========== 平台整体经营概览 ==========")
df_overview

========== 平台整体经营概览 ==========


,total_orders,total_customers,total_sellers,total_revenue,avg_item_price,earliest_order,latest_order
0,96478,96478,2970,13221498.11,119.98,2016-09-15,2018-08-29


# **Sales rankings of various categories(TOP10)**

In [8]:
RANK = """
SELECT
    p.product_category_name_english           AS category,
    COUNT(DISTINCT oi.order_id)               AS total_orders,
    ROUND(SUM(oi.price), 2)                   AS total_revenue,
    ROUND(AVG(oi.price), 2)                   AS avg_price,
    RANK() OVER (ORDER BY SUM(oi.price) DESC) AS revenue_rank
FROM fact_order_items oi
JOIN dim_products p ON oi.product_id = p.product_id
JOIN dim_orders o   ON oi.order_id   = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name_english
ORDER BY revenue_rank
LIMIT 10
"""

df_RANK = pd.read_sql(text(RANK), engine)
df_RANK

,category,total_orders,total_revenue,avg_price,revenue_rank
0,health_beauty,8647,1233131.72,130.28,1
1,watches_gifts,5495,1166176.98,199.04,2
2,bed_bath_table,9272,1023434.76,93.44,3
3,sports_leisure,7530,954852.55,113.25,4
4,computers_accessories,6530,888724.61,116.26,5
5,furniture_decor,6307,711927.69,87.25,6
6,housewares,5743,615628.69,90.60,7
7,cool_stuff,3559,610204.10,164.12,8
8,auto,3810,578966.65,139.85,9
9,toys,3804,471286.48,116.94,10


# **Seller Performance Rating**

In [19]:
seller = """
SELECT
    seller_grade,
    COUNT(*)                        AS seller_count,
    ROUND(AVG(total_orders), 1)     AS avg_orders,
    ROUND(AVG(total_revenue), 2)    AS avg_revenue,
    ROUND(AVG(avg_review_score), 2) AS avg_score
FROM (
    SELECT
        s.seller_id,
        COUNT(DISTINCT oi.order_id)           AS total_orders,
        ROUND(SUM(oi.price), 2)               AS total_revenue,
        ROUND(AVG(r.review_score), 2)         AS avg_review_score,
        CASE
            WHEN AVG(r.review_score) >= 4.5
             AND COUNT(DISTINCT oi.order_id) >= 50  THEN 'S: Star Seller'
            WHEN AVG(r.review_score) >= 4.0
             AND COUNT(DISTINCT oi.order_id) >= 20  THEN 'A: Quality Seller'
            WHEN AVG(r.review_score) >= 3.0         THEN 'B: Average Seller'
            ELSE                                         'C: At-Risk Seller'
        END AS seller_grade
    FROM dim_sellers s
    JOIN fact_order_items oi ON s.seller_id = oi.seller_id
    JOIN dim_orders o        ON oi.order_id  = o.order_id
    JOIN dim_reviews r       ON o.order_id   = r.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY s.seller_id
)
GROUP BY seller_grade
ORDER BY seller_grade
"""

df_seller = pd.read_sql(text(seller), engine)
df_seller

,seller_grade,seller_count,avg_orders,avg_revenue,avg_score
0,A: Quality Seller,521,104.0,14041.87,4.29
1,B: Average Seller,2230,17.8,2357.81,4.30
2,C: At-Risk Seller,186,5.7,1391.07,1.90
3,S: Star Seller,28,75.2,11995.36,4.60


# **Analysis of Repeat Purchase Customers**

In [4]:
sql_repurchase = """
SELECT
    CASE
        WHEN days_between_orders <= 30  THEN 'A: Within 1 month'
        WHEN days_between_orders <= 90  THEN 'B: 1-3 months'
        WHEN days_between_orders <= 180 THEN 'C: 3-6 months'
        WHEN days_between_orders <= 365 THEN 'D: 6-12 months'
        ELSE                                 'E: Over 1 year'
    END AS repurchase_segment,
    COUNT(DISTINCT customer_unique_id)  AS customer_count,
    ROUND(AVG(days_between_orders), 1)  AS avg_days
FROM (
    SELECT
        c.customer_unique_id,
        JULIANDAY(o.order_purchase_timestamp) -
        JULIANDAY(LAG(o.order_purchase_timestamp)
            OVER (PARTITION BY c.customer_unique_id
                  ORDER BY o.order_purchase_timestamp)) AS days_between_orders
    FROM dim_orders o
    JOIN dim_customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
)
WHERE days_between_orders IS NOT NULL
GROUP BY repurchase_segment
ORDER BY repurchase_segment
"""

df_repurchase = pd.read_sql(text(sql_repurchase), engine)
df_repurchase

,repurchase_segment,customer_count,avg_days
0,A: Within 1 month,1462,5.4
1,B: 1-3 months,560,55.6
2,C: 3-6 months,447,131.4
3,D: 6-12 months,426,253.5
4,E: Over 1 year,84,436.8
